# Constraint programming

In [1]:
from constraint import *
from itertools import product

num_students = 20
students = range(num_students)
partners = range(num_students)

problem = Problem()
sp = list(product(students, partners))
variables = problem.addVariables(sp, [0, 1])

# Set constraints
# A student can pick only one partner
for s in students:
    pairs = [(s, p) for p in partners]
    problem.addConstraint(ExactSumConstraint(1), pairs)

# Partners should be symmetric
for s in students:
    for p in partners:
        if s < p:
            pairs = [(s, p), (p, s)]
            problem.addConstraint(AllEqualConstraint(), pairs)

# Some students have formed a pair already
fixe_pairs = [(2, 3), (5, 6), (10, 12), (14, 17), (18, 19)]
for pair in fixe_pairs:
    problem.addConstraint(ExactSumConstraint(1), [pair])

# Some students want to work alone
loners = [0, 1, 4, 7]
loner_keys = [(l, l) for l in loners]
for l in loner_keys:
    problem.addConstraint(ExactSumConstraint(1), [l])

# Print out the solutions
solutions = problem.getSolutions()
print(f"The size of feasible solutions is: {len(solutions)}")

for sol in solutions:
    sol = list(filter(lambda item: item[1] == 1, sol.items()))
    print(sol)

The size of feasible solutions is: 76
[((2, 3), 1), ((3, 2), 1), ((5, 6), 1), ((6, 5), 1), ((10, 12), 1), ((12, 10), 1), ((14, 17), 1), ((17, 14), 1), ((18, 19), 1), ((19, 18), 1), ((8, 9), 1), ((9, 8), 1), ((11, 13), 1), ((13, 11), 1), ((15, 16), 1), ((16, 15), 1), ((0, 0), 1), ((1, 1), 1), ((4, 4), 1), ((7, 7), 1)]
[((2, 3), 1), ((3, 2), 1), ((5, 6), 1), ((6, 5), 1), ((10, 12), 1), ((12, 10), 1), ((14, 17), 1), ((17, 14), 1), ((18, 19), 1), ((19, 18), 1), ((8, 9), 1), ((9, 8), 1), ((11, 13), 1), ((13, 11), 1), ((0, 0), 1), ((1, 1), 1), ((4, 4), 1), ((7, 7), 1), ((15, 15), 1), ((16, 16), 1)]
[((2, 3), 1), ((3, 2), 1), ((5, 6), 1), ((6, 5), 1), ((10, 12), 1), ((12, 10), 1), ((14, 17), 1), ((17, 14), 1), ((18, 19), 1), ((19, 18), 1), ((8, 9), 1), ((9, 8), 1), ((11, 15), 1), ((15, 11), 1), ((13, 16), 1), ((16, 13), 1), ((0, 0), 1), ((1, 1), 1), ((4, 4), 1), ((7, 7), 1)]
[((2, 3), 1), ((3, 2), 1), ((5, 6), 1), ((6, 5), 1), ((10, 12), 1), ((12, 10), 1), ((14, 17), 1), ((17, 14), 1), ((18, 

# Integer programming: Soft constraint

In [2]:
import pulp
from itertools import product

students = [
    {"id": 0, "program": "master"},
    {"id": 1, "program": "master"},
    {"id": 2, "program": "master"},
    {"id": 3, "program": "master"},
    {"id": 4, "program": "master"},
    {"id": 5, "program": "master"},
    {"id": 6, "program": "master"},
    {"id": 7, "program": "master"},
    {"id": 8, "program": "master"},
    {"id": 9, "program": "master"},
    {"id": 10, "program": "phd"},
    {"id": 11, "program": "phd"},
    {"id": 12, "program": "phd"},
    {"id": 13, "program": "phd"},
    {"id": 14, "program": "phd"},
    {"id": 15, "program": "phd"},
    {"id": 16, "program": "phd"},
    {"id": 17, "program": "phd"},
    {"id": 18, "program": "phd"},
    {"id": 19, "program": "phd"},
]
sids = [s["id"] for s in students]
ss = list(product(sids, sids))

previous_pairs = [
    (0, 10),
    (1, 13),
    (2, 3),
    (4, 5),
    (6, 7),
    (8, 19),
    (9, 12),
    (11, 18),
    (14, 17),
    (15, 16)
]

In [3]:
prob = pulp.LpProblem('TeamMaking-1', pulp.LpMaximize)
pair = pulp.LpVariable.dicts('pair', ss, cat='Binary')
# Constraints
# Each student can only find one partner
for sid in sids:
    prob += pulp.lpSum([pair[sid, sid2] for sid2 in sids]) == 1

# The partnership needs to be symmetric
for _ss in ss:
    prob += pair[_ss[0], _ss[1]] - pair[_ss[1], _ss[0]] == 0


# Objective
# Preference
def weight(sid1, sid2):
    reward = 0

    # It is nice if phd student and non-phd student work together
    program1 = students[sid1]['program']
    program2 = students[sid2]['program']
    if program1 != program2:
        reward += 1

    # It is nice if students work together with another student
    if sid1 != sid2:
        reward += 1

    # It is nice if students form a new pair
    if (sid1, sid2) in previous_pairs:
        reward -= 1

    return reward


prob += pulp.lpSum([pair[sid1, sid2] * weight(sid1, sid2) for sid1, sid2 in ss])
# Solve
status = prob.solve()
pulp.LpStatus[status]

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/pulp/apis/../solverdir/cbc/osx/64/cbc /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/04af57453bfe4068aec789ed0cb338d6-pulp.mps max timeMode elapsed branch printingOptions all solution /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/04af57453bfe4068aec789ed0cb338d6-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 425 COLUMNS
At line 2780 RHS
At line 3201 BOUNDS
At line 3602 ENDATA
Problem MODEL has 420 rows, 400 columns and 1160 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 40 - 0.00 seconds
Cgl0004I processed model has 20 rows, 210 columns (210 integer (210 of which binary)) and 400 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solut

'Optimal'

In [4]:
for key in ss:
    if pair[key].value() > 0:
        print(pair[key])
        print(students[key[0]])
        print(students[key[1]])
        print()

pair_(0,_15)
{'id': 0, 'program': 'master'}
{'id': 15, 'program': 'phd'}

pair_(1,_14)
{'id': 1, 'program': 'master'}
{'id': 14, 'program': 'phd'}

pair_(2,_12)
{'id': 2, 'program': 'master'}
{'id': 12, 'program': 'phd'}

pair_(3,_17)
{'id': 3, 'program': 'master'}
{'id': 17, 'program': 'phd'}

pair_(4,_18)
{'id': 4, 'program': 'master'}
{'id': 18, 'program': 'phd'}

pair_(5,_11)
{'id': 5, 'program': 'master'}
{'id': 11, 'program': 'phd'}

pair_(6,_16)
{'id': 6, 'program': 'master'}
{'id': 16, 'program': 'phd'}

pair_(7,_10)
{'id': 7, 'program': 'master'}
{'id': 10, 'program': 'phd'}

pair_(8,_13)
{'id': 8, 'program': 'master'}
{'id': 13, 'program': 'phd'}

pair_(9,_19)
{'id': 9, 'program': 'master'}
{'id': 19, 'program': 'phd'}

pair_(10,_7)
{'id': 10, 'program': 'phd'}
{'id': 7, 'program': 'master'}

pair_(11,_5)
{'id': 11, 'program': 'phd'}
{'id': 5, 'program': 'master'}

pair_(12,_2)
{'id': 12, 'program': 'phd'}
{'id': 2, 'program': 'master'}

pair_(13,_8)
{'id': 13, 'program': 'ph

# Integer Programming: Hard Constraints

In [6]:
prob2 = pulp.LpProblem('TeamMaking-2', pulp.LpMaximize)
pair2 = pulp.LpVariable.dicts('pair', ss, cat='Binary')
# Constraints
# Each student can only find one partner
for sid in sids:
    prob2 += pulp.lpSum([pair2[sid, sid2] for sid2 in sids]) == 1

# The partnership needs to be symmetric
for _ss in ss:
    prob2 += pair2[_ss[0], _ss[1]] - pair2[_ss[1], _ss[0]] == 0


# Students are not allowed to work with the previous partner
# TODO
for prev_pair in previous_pairs:
    prob += pair[prev_pair[0], prev_pair[1]] == 0

# Objective
# Preference
def weight(sid1, sid2):
    reward = 0

    # It is nice if phd student and non-phd student work together
    program1 = students[sid1]['program']
    program2 = students[sid2]['program']
    if program1 != program2:
        reward += 1

    # It is nice if students work together with another student
    if sid1 != sid2:
        reward += 1

    return reward


prob2 += pulp.lpSum([pair2[sid1, sid2] * weight(sid1, sid2) for sid1, sid2 in ss])
status = prob2.solve()
pulp.LpStatus[status]

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/kotarohara/repo/Python/web-optimization/venv/lib/python3.9/site-packages/pulp/apis/../solverdir/cbc/osx/64/cbc /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/7d80d775fd844ef9afdfbbe40ad8d56b-pulp.mps max timeMode elapsed branch printingOptions all solution /var/folders/4_/vrr8kzqn5b9dxsprxn13022m0000gn/T/7d80d775fd844ef9afdfbbe40ad8d56b-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 425 COLUMNS
At line 2786 RHS
At line 3207 BOUNDS
At line 3608 ENDATA
Problem MODEL has 420 rows, 400 columns and 1160 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 40 - 0.00 seconds
Cgl0004I processed model has 20 rows, 210 columns (210 integer (210 of which binary)) and 400 elements
Cutoff increment increased from 1e-05 to 1.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solut

'Optimal'

In [7]:
for key in ss:
    if pair[key].value() > 0:
        print(pair[key])
        print(students[key[0]])
        print(students[key[1]])
        print()

pair_(0,_15)
{'id': 0, 'program': 'master'}
{'id': 15, 'program': 'phd'}

pair_(1,_14)
{'id': 1, 'program': 'master'}
{'id': 14, 'program': 'phd'}

pair_(2,_12)
{'id': 2, 'program': 'master'}
{'id': 12, 'program': 'phd'}

pair_(3,_17)
{'id': 3, 'program': 'master'}
{'id': 17, 'program': 'phd'}

pair_(4,_18)
{'id': 4, 'program': 'master'}
{'id': 18, 'program': 'phd'}

pair_(5,_11)
{'id': 5, 'program': 'master'}
{'id': 11, 'program': 'phd'}

pair_(6,_16)
{'id': 6, 'program': 'master'}
{'id': 16, 'program': 'phd'}

pair_(7,_10)
{'id': 7, 'program': 'master'}
{'id': 10, 'program': 'phd'}

pair_(8,_13)
{'id': 8, 'program': 'master'}
{'id': 13, 'program': 'phd'}

pair_(9,_19)
{'id': 9, 'program': 'master'}
{'id': 19, 'program': 'phd'}

pair_(10,_7)
{'id': 10, 'program': 'phd'}
{'id': 7, 'program': 'master'}

pair_(11,_5)
{'id': 11, 'program': 'phd'}
{'id': 5, 'program': 'master'}

pair_(12,_2)
{'id': 12, 'program': 'phd'}
{'id': 2, 'program': 'master'}

pair_(13,_8)
{'id': 13, 'program': 'ph